<a href="https://colab.research.google.com/github/26b21a4255kiet-cmd/PHEONIX/blob/main/IMAGE_TRANSFORMATION_DECOMPOSER_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# IMAGE TRANSFORMATION DECOMPOSER (B10)
# B.Tech Mathematics Mini Project
#
# Concepts:
#   1. Eigenvalues
#   2. Eigenvectors
#   3. Polar Decomposition A = Q S
#   4. Rotation / Orthogonal Transformation
#   5. Stretching
#   6. Image Transformation
#   7. Geometric Interpretation
#
# Environment:
#   Google Colab + Python
#   NumPy + SymPy + Matplotlib + PIL
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import files

np.set_printoptions(precision=4, suppress=True)

print("=" * 70)
print("       IMAGE TRANSFORMATION DECOMPOSER (B10)")
print("=" * 70)


# ================================================================
# 2. USER INPUT
# ================================================================
#
# Change this matrix to test different transformations.
#
# Example:
#
# A = [[1.2, 0.4],
#      [0.2, 1.5]]
#
# Mathematical transformation:
#
#       [x']       [a b] [x]
#       [y']   =   [c d] [y]
#
# ================================================================

A = np.array([
    [1.2, 0.4],
    [0.2, 1.5]
], dtype=float)


# Set this to True if you want to upload your own image.
# Set False to use the automatically generated test image.

USE_UPLOADED_IMAGE = False

TEST_IMAGE_SIZE = 500


# ================================================================
# 3. INPUT VALIDATION
# ================================================================

if A.shape != (2, 2):
    raise ValueError("The transformation matrix must be 2 × 2.")

if not np.all(np.isfinite(A)):
    raise ValueError("Matrix contains invalid values.")

if abs(np.linalg.det(A)) < 1e-12:
    print("\nWARNING:")
    print("The matrix is singular.")
    print("Some image transformation features require an invertible matrix.")


print("\nInput Transformation Matrix A:")
print(A)


# ================================================================
# 4. EIGENVALUE AND EIGENVECTOR CALCULATION
# ================================================================

print("\n" + "=" * 70)
print("1. EIGENVALUE AND EIGENVECTOR ANALYSIS")
print("=" * 70)

# Convert NumPy matrix to SymPy matrix
A_sym = sp.Matrix(A)

print("\nMatrix A:")
sp.pprint(A_sym)


# Characteristic equation
lam = sp.symbols("λ")

characteristic_polynomial = A_sym.charpoly(lam).as_expr()

print("\nCharacteristic Polynomial:")
sp.pprint(characteristic_polynomial)


# Eigenvalues
sym_eigenvalues = A_sym.eigenvals()

print("\nEigenvalues:")

for value, multiplicity in sym_eigenvalues.items():
    print(f"λ = {value}   multiplicity = {multiplicity}")


# Eigenvectors
sym_eigenvectors = A_sym.eigenvects()

print("\nEigenvectors:")

for eigenvalue, multiplicity, vectors in sym_eigenvectors:

    print(f"\nFor λ = {eigenvalue}")

    for vector in vectors:
        sp.pprint(vector)


# Numerical eigenvalues/eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A)

print("\nNumerical Eigenvalues:")

for i, value in enumerate(eigenvalues):

    print(
        f"λ{i+1} = "
        f"{value.real:.6f}"
        +
        (f" + {value.imag:.6f}i"
         if abs(value.imag) > 1e-10 else "")
    )


print("\nNumerical Eigenvectors:")

for i in range(2):

    print(
        f"v{i+1} = "
        f"{eigenvectors[:, i]}"
    )


print("""
Mathematical meaning:

For an eigenvector v,

        A v = λ v

Therefore, the transformation changes the magnitude
of v by λ without changing its line direction.
""")


# ================================================================
# 5. POLAR DECOMPOSITION
# ================================================================
#
# A = Q S
#
# Q = orthogonal part
# S = symmetric positive-semidefinite stretching part
#
# We use SVD:
#
# A = U Σ Vᵀ
#
# Q = U Vᵀ
#
# S = V Σ Vᵀ
#
# ================================================================

print("\n" + "=" * 70)
print("2. POLAR DECOMPOSITION")
print("=" * 70)

if abs(np.linalg.det(A)) < 1e-12:

    print("\nPolar decomposition visualization skipped")
    print("because the matrix is singular.")

else:

    U, singular_values, Vt = np.linalg.svd(A)

    Q = U @ Vt

    S = Vt.T @ np.diag(singular_values) @ Vt

    print("\nOrthogonal / Rotation Part Q:")
    print(Q)

    print("\nStretching Part S:")
    print(S)

    print("\nOriginal Matrix A:")
    print(A)

    print("\nReconstructed Q × S:")
    print(Q @ S)

    reconstruction_error = np.linalg.norm(A - Q @ S)

    print("\nReconstruction Error:")
    print(f"||A - QS|| = {reconstruction_error:.12f}")


    # Determine rotation/reflection
    det_Q = np.linalg.det(Q)

    print("\ndeterminant(Q) =", det_Q)

    if det_Q > 0:

        print("Q represents a pure rotation.")

        rotation_angle = np.arctan2(
            Q[1, 0],
            Q[0, 0]
        )

        rotation_degrees = np.degrees(
            rotation_angle
        )

        print(
            f"Rotation angle = "
            f"{rotation_degrees:.4f}°"
        )

    else:

        print(
            "Q contains a reflection in addition "
            "to an orthogonal transformation."
        )


# ================================================================
# 6. PRINCIPAL STRETCHING DIRECTIONS
# ================================================================

print("\n" + "=" * 70)
print("3. PRINCIPAL STRETCHING DIRECTIONS")
print("=" * 70)

if abs(np.linalg.det(A)) >= 1e-12:

    stretch_values, stretch_vectors = np.linalg.eigh(S)

    print("\nStretch Factors:")

    for i, value in enumerate(stretch_values):

        print(
            f"σ{i+1} = "
            f"{value:.6f}"
        )


    print("\nPrincipal Stretching Directions:")

    for i in range(2):

        print(
            f"u{i+1} = "
            f"{stretch_vectors[:, i]}"
        )


    print("""
Mathematical meaning:

For an eigenvector u of S,

        S u = σ u

Therefore, along the direction u,
S produces only stretching/compression.

σ > 1  → stretching
σ = 1  → no change
σ < 1  → compression
""")


# ================================================================
# 7. CREATE / UPLOAD IMAGE
# ================================================================

print("\n" + "=" * 70)
print("4. IMAGE INPUT")
print("=" * 70)

if USE_UPLOADED_IMAGE:

    print("\nUpload an image:")

    uploaded = files.upload()

    filename = next(iter(uploaded))

    original_image = Image.open(
        filename
    ).convert("RGB")

    print(
        f"Image loaded: {filename}"
    )

else:

    print(
        "\nNo image uploaded."
    )

    print(
        "Creating mathematical test image..."
    )

    n = TEST_IMAGE_SIZE

    y, x = np.mgrid[
        -1:1:complex(n),
        -1:1:complex(n)
    ]

    test = np.ones(
        (n, n, 3),
        dtype=np.uint8
    ) * 255


    # Circle
    circle_mask = (
        x**2 + y**2 < 0.55**2
    )

    test[circle_mask] = [
        80, 140, 220
    ]


    # Square
    square_mask = (
        (x > -0.25) &
        (x < 0.25) &
        (y > -0.25) &
        (y < 0.25)
    )

    test[square_mask] = [
        220, 100, 100
    ]


    # Coordinate axes
    axis_mask = (
        (np.abs(x) < 0.008) |
        (np.abs(y) < 0.008)
    )

    test[axis_mask] = [
        30, 30, 30
    ]


    original_image = Image.fromarray(
        test
    )

    print(
        "Mathematical test image created."
    )


# ================================================================
# 8. IMAGE TRANSFORMATION FUNCTION
# ================================================================

def transform_image(
    image,
    matrix,
    output_size=None
):

    """
    Transform an image using a 2 × 2 matrix.

    Coordinate transformation:

          [x']       [a b] [x]
          [y']  =    [c d] [y]

    Inverse mapping is used so that
    the transformed image does not have
    unnecessary holes.
    """

    image = image.convert("RGB")

    if output_size is None:

        output_size = image.size


    width, height = output_size

    image_array = np.asarray(
        image
    )


    # Output coordinates
    yy, xx = np.mgrid[
        0:height,
        0:width
    ]


    # Center coordinates
    x = xx - width / 2

    y = -(yy - height / 2)


    transformed_coordinates = np.vstack([
        x.ravel(),
        y.ravel()
    ])


    # Inverse mapping
    determinant = np.linalg.det(matrix)

    if abs(determinant) < 1e-12:

        raise ValueError(
            "Cannot perform inverse image mapping "
            "because matrix is singular."
        )


    inverse_matrix = np.linalg.inv(
        matrix
    )


    original_coordinates = (
        inverse_matrix
        @ transformed_coordinates
    )


    original_x = (
        original_coordinates[0, :]
        + image.width / 2
    )

    original_y = (
        -original_coordinates[1, :]
        + image.height / 2
    )


    # Nearest-neighbour sampling
    original_x = np.round(
        original_x
    ).astype

       IMAGE TRANSFORMATION DECOMPOSER (B10)

Input Transformation Matrix A:
[[1.2 0.4]
 [0.2 1.5]]

1. EIGENVALUE AND EIGENVECTOR ANALYSIS

Matrix A:
⎡1.2  0.4⎤
⎢        ⎥
⎣0.2  1.5⎦

Characteristic Polynomial:
     2               
1.0⋅λ  - 2.7⋅λ + 1.72

Eigenvalues:
λ = 1.02984378812836   multiplicity = 1
λ = 1.67015621187164   multiplicity = 1

Eigenvectors:

For λ = 1.02984378812836
⎡-0.92020150397314⎤
⎢                 ⎥
⎣0.391445005186644⎦

For λ = 1.67015621187164
⎡-0.678867666328721⎤
⎢                  ⎥
⎣-0.797934625908134⎦

Numerical Eigenvalues:
λ1 = 1.029844
λ2 = 1.670156

Numerical Eigenvectors:
v1 = [-0.9202  0.3914]
v2 = [-0.648  -0.7616]

Mathematical meaning:

For an eigenvector v,

        A v = λ v

Therefore, the transformation changes the magnitude
of v by λ without changing its line direction.


2. POLAR DECOMPOSITION

Orthogonal / Rotation Part Q:
[[ 0.9973  0.0739]
 [-0.0739  0.9973]]

Stretching Part S:
[[1.1819 0.2881]
 [0.2881 1.5255]]

Original Matrix A:
[[